# Orbit Wars Agent Submission

## Goal

Package the current Orbit Wars agent candidate, run a fixed-seed smoke benchmark on Kaggle, and write a submission archive.

This notebook is intentionally version-agnostic. Keep agent history and decisions in `docs/04_agent_version_log.md`; change only `CFG["agent_version"]` when promoting a new candidate.

## Competition Contract

Each turn, the agent receives an observation with **planets**, **fleets**, our **player id**, and orbit metadata such as **angular velocity**. The submitted `agent(obs)` must return moves in the form `[from_planet_id, angle_in_radians, num_ships]`.

## Decision Model

The active agent treats each owned planet as an independent **source decision**. For each source, it builds affordable candidate targets, rejects unsafe launches, scores the remaining candidates with **production**, **ship cost**, **travel time**, **sun risk**, and **defense state**, then emits one move at most. This mirrors the useful structure from RL tutorials while keeping the current submission deterministic and inspectable.

The smoke benchmark is not a leaderboard proxy; it is an **acceptance gate** for packaging, environment compatibility, action format, and obvious strategy regressions before submission. Strategy lessons and score history stay in the docs, not in this notebook.

## Artifacts Written

- `main.py`
- `submission.tar.gz`
- `agent_submission_environment_info.json`
- `agent_submission_seed_summary.csv`
- `agent_submission_run_errors.csv`
- `agent_submission_findings.md`


## 1. Runtime Setup

Select the active agent version, resolve its source, and write a root-level `main.py` in the Kaggle working directory. The notebook is **self-contained** for Kaggle execution by embedding the promoted agent source, while local runs can still read from `agents/<agent_version>/main.py`.


In [ ]:
import importlib.util
import json
import platform
import shutil
import sys
import tarfile
import traceback
from pathlib import Path
from typing import Any, Optional, Sequence

import pandas as pd

EMBEDDED_AGENT_VERSION = 'roi_reserve_v5'
EMBEDDED_AGENT_SOURCE = '"""Combat-survival, reinforcement-capable, orbit-aware, and sun-safe Orbit Wars agent v5."""\n\nimport math\nfrom collections import namedtuple\nfrom typing import Any, Iterable, Optional\n\ntry:\n    from kaggle_environments.envs.orbit_wars.orbit_wars import Fleet, Planet\nexcept Exception:\n    Planet = namedtuple("Planet", "id owner x y radius ships production")\n    Fleet = namedtuple("Fleet", "id owner x y angle from_planet_id ships")\n\n\nBOARD_CENTER = (50.0, 50.0)\nSUN_RADIUS = 10.0\nSUN_SAFETY_MARGIN = 0.25\nMAX_FLEET_SPEED = 6.0\nEARLY_OWNED_COUNT = 3\nTHREAT_HORIZON_TURNS = 30\nPLANET_THREAT_MARGIN = 0.75\nREINFORCE_LOOKAHEAD_TURNS = 35\nREINFORCE_SAFETY_MARGIN = 2\nENEMY_PRODUCTION_MARGIN = 2\nSOURCE_SURVIVAL_LOOKAHEAD_TURNS = 45\nSOURCE_SURVIVAL_MARGIN = 2\nCONTESTED_TARGET_WINDOW = 10.0\nMULTIPLAYER_STABLE_PRODUCTION = 30\n\n\ndef get_obs_value(obs: Any, name: str, default: Any) -> Any:\n    """Read an observation field from either dict or object observations.\n\n    Args:\n        obs: Kaggle observation as a dict-like or object-like value.\n        name: Field name to read.\n        default: Value returned when the field is absent.\n\n    Returns:\n        The observation field value, or `default` when unavailable.\n    """\n    if isinstance(obs, dict):\n        return obs.get(name, default)\n    return getattr(obs, name, default)\n\n\ndef parse_planets(rows: Iterable[Iterable[Any]]) -> list:\n    """Convert raw planet rows to Orbit Wars planet tuples.\n\n    Args:\n        rows: Iterable of `[id, owner, x, y, radius, ships, production]`.\n\n    Returns:\n        Parsed planet records with attribute access.\n    """\n    return [Planet(*row) for row in rows]\n\n\ndef parse_fleets(rows: Iterable[Iterable[Any]]) -> list:\n    """Convert raw fleet rows to Orbit Wars fleet tuples.\n\n    Args:\n        rows: Iterable of `[id, owner, x, y, angle, from_planet_id, ships]`.\n\n    Returns:\n        Parsed fleet records with attribute access.\n    """\n    return [Fleet(*row) for row in rows]\n\n\ndef distance(source: Any, target: Any) -> float:\n    """Calculate Euclidean distance between two planet-like objects."""\n    return math.hypot(float(target.x) - float(source.x), float(target.y) - float(source.y))\n\n\ndef launch_angle(source: Any, target: Any) -> float:\n    """Calculate launch angle from source to target in radians."""\n    return float(math.atan2(float(target.y) - float(source.y), float(target.x) - float(source.x)))\n\n\ndef fleet_speed(ships: int, max_speed: float = MAX_FLEET_SPEED) -> float:\n    """Calculate fleet speed from ship count using the Orbit Wars curve.\n\n    Args:\n        ships: Number of ships in the fleet.\n        max_speed: Environment maximum fleet speed.\n\n    Returns:\n        Fleet speed in board units per turn.\n    """\n    ships = max(int(ships), 1)\n    return 1.0 + (max_speed - 1.0) * (math.log(ships) / math.log(1000.0)) ** 1.5\n\n\ndef total_production(planets: Iterable[Any]) -> int:\n    """Return total production controlled by a planet collection."""\n    return int(sum(int(planet.production) for planet in planets))\n\n\ndef source_reserve(planet: Any, owned_count: int = 1) -> int:\n    """Calculate ships to keep on a source planet before launching.\n\n    Args:\n        planet: Source planet record.\n        owned_count: Number of planets currently owned by the agent.\n\n    Returns:\n        Minimum ships to preserve on the source.\n    """\n    production = float(planet.production)\n    base = int(max(5, production * 3))\n    if owned_count <= 1:\n        if production <= 1:\n            return 4\n        if production <= 3:\n            return max(base, 8)\n        return max(base, 12)\n    if owned_count == 2:\n        return max(base, 10)\n    return base\n\n\ndef point_segment_distance(\n    px: float,\n    py: float,\n    ax: float,\n    ay: float,\n    bx: float,\n    by: float,\n) -> float:\n    """Calculate the shortest distance from a point to a segment."""\n    dx = bx - ax\n    dy = by - ay\n    if dx == 0 and dy == 0:\n        return math.hypot(px - ax, py - ay)\n    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)\n    t = max(0.0, min(1.0, t))\n    closest_x = ax + t * dx\n    closest_y = ay + t * dy\n    return math.hypot(px - closest_x, py - closest_y)\n\n\ndef crosses_sun(source: Any, target: Any) -> bool:\n    """Return whether the direct launch segment intersects the sun."""\n    center_x, center_y = BOARD_CENTER\n    clearance = point_segment_distance(\n        center_x,\n        center_y,\n        float(source.x),\n        float(source.y),\n        float(target.x),\n        float(target.y),\n    )\n    return clearance <= SUN_RADIUS + SUN_SAFETY_MARGIN\n\n\ndef fleet_intersects_planet(fleet: Any, planet: Any, horizon_turns: int = THREAT_HORIZON_TURNS) -> bool:\n    """Return whether a fleet\'s near-term trajectory intersects a planet."""\n    speed = fleet_speed(int(fleet.ships))\n    start_x = float(fleet.x)\n    start_y = float(fleet.y)\n    end_x = start_x + math.cos(float(fleet.angle)) * speed * horizon_turns\n    end_y = start_y + math.sin(float(fleet.angle)) * speed * horizon_turns\n    clearance = point_segment_distance(\n        float(planet.x),\n        float(planet.y),\n        start_x,\n        start_y,\n        end_x,\n        end_y,\n    )\n    return clearance <= float(planet.radius) + PLANET_THREAT_MARGIN\n\n\ndef fleet_eta_to_planet(fleet: Any, planet: Any, horizon_turns: int = REINFORCE_LOOKAHEAD_TURNS) -> Optional[int]:\n    """Estimate when a fleet trajectory first reaches a planet.\n\n    Args:\n        fleet: Fleet-like record with position, angle, and ship count.\n        planet: Planet-like collision target.\n        horizon_turns: Maximum ETA to consider.\n\n    Returns:\n        ETA in turns, or `None` if the fleet does not hit within the horizon.\n    """\n    speed = fleet_speed(int(fleet.ships))\n    dir_x = math.cos(float(fleet.angle))\n    dir_y = math.sin(float(fleet.angle))\n    dx = float(planet.x) - float(fleet.x)\n    dy = float(planet.y) - float(fleet.y)\n    projection = dx * dir_x + dy * dir_y\n    if projection < 0:\n        return None\n    perpendicular_sq = dx * dx + dy * dy - projection * projection\n    radius = float(planet.radius) + PLANET_THREAT_MARGIN\n    if perpendicular_sq > radius * radius:\n        return None\n    hit_distance = max(0.0, projection - math.sqrt(max(0.0, radius * radius - perpendicular_sq)))\n    eta = int(math.ceil(hit_distance / speed))\n    if eta < 1 or eta > horizon_turns:\n        return None\n    return eta\n\n\ndef incoming_reinforcement_needs(player: int, planets: list, fleets: list) -> dict:\n    """Compute owned planets that need timed reinforcement to survive visible fleets."""\n    needs = {}\n    owned_planets = [planet for planet in planets if int(planet.owner) == player]\n    for planet in owned_planets:\n        enemy_arrivals = []\n        for fleet in fleets:\n            if int(fleet.owner) == player:\n                continue\n            eta = fleet_eta_to_planet(fleet, planet)\n            if eta is None:\n                continue\n            enemy_arrivals.append((eta, int(fleet.ships)))\n        if not enemy_arrivals:\n            continue\n\n        enemy_arrivals.sort()\n        garrison = float(planet.ships)\n        previous_turn = 0\n        for eta, ships in enemy_arrivals:\n            garrison += max(0, eta - previous_turn) * float(planet.production)\n            garrison -= ships\n            previous_turn = eta\n            if garrison < 0:\n                needs[int(planet.id)] = {\n                    "planet": planet,\n                    "ships_needed": int(math.ceil(-garrison)) + REINFORCE_SAFETY_MARGIN,\n                    "needed_by_turn": eta,\n                }\n                break\n    return needs\n\n\ndef enemy_players(player: int, planets: list, fleets: list) -> set:\n    """Return visible non-neutral enemy owners."""\n    owners = {\n        int(planet.owner)\n        for planet in planets\n        if int(planet.owner) >= 0 and int(planet.owner) != player\n    }\n    owners.update(\n        int(fleet.owner)\n        for fleet in fleets\n        if int(fleet.owner) >= 0 and int(fleet.owner) != player\n    )\n    return owners\n\n\ndef threatened_planet_ids(player: int, planets: list, fleets: list) -> set:\n    """Identify owned planets whose launch budget should be frozen for defense."""\n    owned_planets = [planet for planet in planets if int(planet.owner) == player]\n    threats = set()\n    for fleet in fleets:\n        if int(fleet.owner) == player:\n            continue\n        for planet in owned_planets:\n            if int(fleet.ships) < source_reserve(planet, len(owned_planets)):\n                continue\n            if fleet_intersects_planet(fleet, planet):\n                threats.add(int(planet.id))\n    return threats\n\n\ndef projected_owned_garrison(\n    planet: Any,\n    player: int,\n    fleets: list,\n    ships_spent: int = 0,\n    horizon_turns: int = SOURCE_SURVIVAL_LOOKAHEAD_TURNS,\n) -> float:\n    """Project owned-planet garrison after visible fleet arrivals."""\n    arrivals = []\n    for fleet in fleets:\n        eta = fleet_eta_to_planet(fleet, planet, horizon_turns)\n        if eta is None:\n            continue\n        arrivals.append((eta, int(fleet.owner), int(fleet.ships)))\n    arrivals.sort()\n\n    garrison = float(planet.ships) - float(ships_spent)\n    previous_turn = 0\n    for eta, owner, ships in arrivals:\n        garrison += max(0, eta - previous_turn) * float(planet.production)\n        if owner == player:\n            garrison += ships\n        else:\n            garrison -= ships\n        previous_turn = eta\n        if garrison < 0:\n            return garrison\n    return garrison\n\n\ndef source_survives_launch(\n    source: Any,\n    player: int,\n    fleets: list,\n    ships: int,\n    owned_count: int,\n) -> bool:\n    """Return whether a source remains safe after spending launch ships."""\n    projected = projected_owned_garrison(source, player, fleets, ships)\n    reserve_floor = source_reserve(source, owned_count) + SOURCE_SURVIVAL_MARGIN\n    return projected >= reserve_floor\n\n\ndef is_orbiting_planet(planet: Any) -> bool:\n    """Return whether a planet is inside the orbiting-radius threshold."""\n    orbital_radius = math.hypot(float(planet.x) - BOARD_CENTER[0], float(planet.y) - BOARD_CENTER[1])\n    return orbital_radius + float(planet.radius) < 50.0\n\n\ndef rotate_point(x: float, y: float, radians: float) -> tuple[float, float]:\n    """Rotate a point around the Orbit Wars board center.\n\n    Args:\n        x: Current x coordinate.\n        y: Current y coordinate.\n        radians: Rotation angle.\n\n    Returns:\n        Rotated `(x, y)` coordinate.\n    """\n    center_x, center_y = BOARD_CENTER\n    rel_x = x - center_x\n    rel_y = y - center_y\n    cos_a = math.cos(radians)\n    sin_a = math.sin(radians)\n    return (\n        center_x + rel_x * cos_a - rel_y * sin_a,\n        center_y + rel_x * sin_a + rel_y * cos_a,\n    )\n\n\ndef predicted_target(source: Any, target: Any, ships: int, angular_velocity: float) -> Any:\n    """Predict a moving target position near fleet arrival time.\n\n    Args:\n        source: Source planet.\n        target: Target planet at the current observation.\n        ships: Ships planned for launch.\n        angular_velocity: Environment angular velocity in radians per turn.\n\n    Returns:\n        Target-like record with future x/y coordinates when prediction applies.\n    """\n    if not angular_velocity or not is_orbiting_planet(target):\n        return target\n    arrival_turns = distance(source, target) / fleet_speed(ships)\n    future_x, future_y = rotate_point(float(target.x), float(target.y), angular_velocity * arrival_turns)\n    return Planet(\n        int(target.id),\n        int(target.owner),\n        future_x,\n        future_y,\n        float(target.radius),\n        int(target.ships),\n        int(target.production),\n    )\n\n\ndef enemy_support_pressure(\n    player: int,\n    target: Any,\n    planets: list,\n    travel_turns: float,\n) -> int:\n    """Estimate visible enemy support that can contest a target near arrival."""\n    if int(target.owner) < 0:\n        return 0\n\n    pressure = 0.0\n    for planet in planets:\n        if int(planet.owner) < 0 or int(planet.owner) == player or int(planet.id) == int(target.id):\n            continue\n        support_eta = distance(planet, target) / fleet_speed(max(int(planet.ships), 1))\n        if support_eta > travel_turns + CONTESTED_TARGET_WINDOW:\n            continue\n        pressure += float(planet.production) * max(1.0, travel_turns - support_eta + CONTESTED_TARGET_WINDOW)\n        pressure += max(0.0, float(planet.ships) - source_reserve(planet, 4)) * 0.35\n    return int(math.ceil(pressure))\n\n\ndef capture_ships_needed(source: Any, target: Any, seed_ships: int, player: int, planets: list) -> int:\n    """Estimate capture cost at fleet arrival time."""\n    needed = int(target.ships) + 1\n    if int(target.owner) < 0:\n        return needed\n\n    ships = max(needed, int(seed_ships))\n    for _ in range(3):\n        travel_turns = distance(source, target) / fleet_speed(ships)\n        support = enemy_support_pressure(player, target, planets, travel_turns)\n        next_needed = (\n            int(math.ceil(int(target.ships) + travel_turns * float(target.production)))\n            + support\n            + ENEMY_PRODUCTION_MARGIN\n        )\n        if next_needed == ships:\n            break\n        ships = max(needed, next_needed)\n    return int(ships)\n\n\ndef ships_to_send(source: Any, target: Any, owned_count: int, player: int, planets: list, fleets: list) -> Optional[int]:\n    """Calculate a conservative capture fleet while preserving reserve.\n\n    Args:\n        source: Owned source planet.\n        target: Non-owned target planet.\n\n    Returns:\n        Ship count to send, or `None` if the source cannot afford the move.\n    """\n    needed = capture_ships_needed(source, target, int(target.ships) + 1, player, planets)\n    available = int(source.ships) - source_reserve(source, owned_count)\n    if available < needed:\n        return None\n    if not source_survives_launch(source, player, fleets, needed, owned_count):\n        return None\n    return int(needed)\n\n\ndef build_reinforcement_moves(\n    player: int,\n    my_planets: list,\n    planets: list,\n    fleets: list,\n    angular_velocity: float,\n) -> tuple[list, set]:\n    """Create high-priority reinforcement moves for planets forecast to fall."""\n    needs = incoming_reinforcement_needs(player, planets, fleets)\n    if not needs:\n        return [], set()\n\n    moves = []\n    used_sources = set()\n    ordered_needs = sorted(needs.values(), key=lambda item: (item["needed_by_turn"], -int(item["planet"].production)))\n    for need in ordered_needs:\n        target = need["planet"]\n        due_turn = int(need["needed_by_turn"])\n        ships_needed = max(1, int(need["ships_needed"]))\n        best = None\n        for source in my_planets:\n            if int(source.id) == int(target.id) or int(source.id) in used_sources:\n                continue\n            available = int(source.ships) - source_reserve(source, len(my_planets))\n            if available < ships_needed:\n                continue\n            send = min(available, ships_needed)\n            if not source_survives_launch(source, player, fleets, send, len(my_planets)):\n                continue\n            aim_target = predicted_target(source, target, send, angular_velocity)\n            if crosses_sun(source, aim_target):\n                continue\n            travel_turns = distance(source, aim_target) / fleet_speed(send)\n            if travel_turns > due_turn:\n                continue\n            candidate = (travel_turns, -int(source.production), source, aim_target, send)\n            if best is None or candidate < best:\n                best = candidate\n        if best is None:\n            continue\n        _, _, source, aim_target, send = best\n        used_sources.add(int(source.id))\n        moves.append([int(source.id), launch_angle(source, aim_target), int(send)])\n    return moves, used_sources\n\n\ndef target_score(\n    source: Any,\n    target: Any,\n    ships: int,\n    player: int,\n    planets: list,\n    active_enemy_count: int,\n    production_owned: int,\n) -> float:\n    """Score a target by production value adjusted for cost and travel time."""\n    travel_time = distance(source, target) / fleet_speed(ships)\n    capture_cost = max(float(ships), 1.0)\n    production_value = float(target.production) * 30.0\n    enemy_bonus = 16.0 if int(target.owner) >= 0 else 0.0\n    support = enemy_support_pressure(player, target, planets, travel_time)\n    multiplayer_penalty = 0.0\n    if active_enemy_count > 1 and production_owned < MULTIPLAYER_STABLE_PRODUCTION:\n        multiplayer_penalty = 18.0 if int(target.owner) >= 0 else 4.0\n    return (production_value + enemy_bonus) / (capture_cost + support + travel_time + multiplayer_penalty)\n\n\ndef early_travel_cap(owned_count: int, production_owned: int) -> float:\n    """Return the maximum preferred travel time before the economy is stable."""\n    if owned_count <= 1:\n        return 18.0\n    if owned_count <= EARLY_OWNED_COUNT or production_owned < 15:\n        return 24.0\n    return 36.0\n\n\ndef best_target(\n    source: Any,\n    targets: list,\n    reserved_target_ids: set,\n    owned_count: int,\n    angular_velocity: float,\n    production_owned: int,\n    player: int,\n    planets: list,\n    fleets: list,\n    active_enemy_count: int,\n) -> Optional[tuple]:\n    """Choose the best affordable, sun-safe target for one source planet."""\n    candidates = []\n    for target in targets:\n        if int(target.id) in reserved_target_ids:\n            continue\n        if active_enemy_count > 1 and production_owned < MULTIPLAYER_STABLE_PRODUCTION and int(target.owner) >= 0:\n            continue\n        if crosses_sun(source, target):\n            continue\n        ships = ships_to_send(source, target, owned_count, player, planets, fleets)\n        if ships is None:\n            continue\n        aim_target = predicted_target(source, target, ships, angular_velocity)\n        if crosses_sun(source, aim_target):\n            continue\n        travel_time = distance(source, target) / fleet_speed(ships)\n        score = target_score(source, target, ships, player, planets, active_enemy_count, production_owned)\n        cluster_distance = min(\n            (\n                distance(target, planet)\n                for planet in planets\n                if int(planet.owner) == player and int(planet.id) != int(source.id)\n            ),\n            default=distance(source, target),\n        )\n        candidates.append((target, aim_target, ships, travel_time, score, cluster_distance))\n    if not candidates:\n        return None\n\n    if owned_count <= EARLY_OWNED_COUNT or production_owned < 15:\n        cap = early_travel_cap(owned_count, production_owned)\n        local_neutrals = [\n            candidate\n            for candidate in candidates\n            if int(candidate[0].owner) < 0 and candidate[3] <= cap\n        ]\n        if local_neutrals:\n            candidates = local_neutrals\n        else:\n            capped = [candidate for candidate in candidates if candidate[3] <= cap * 1.35]\n            if capped:\n                candidates = capped\n\n    best = max(candidates, key=lambda item: (item[4], -item[3], -item[5], int(item[0].production)))\n    target, aim_target, ships, _, _, _ = best\n    return target, aim_target, ships\n\n\ndef agent(obs: Any) -> list:\n    """Return reinforcement-first, ROI-ranked, reserve-safe Orbit Wars launch actions."""\n    try:\n        player = int(get_obs_value(obs, "player", 0))\n        raw_planets = get_obs_value(obs, "planets", [])\n        raw_fleets = get_obs_value(obs, "fleets", [])\n        angular_velocity = float(get_obs_value(obs, "angular_velocity", 0.0) or 0.0)\n        planets = parse_planets(raw_planets)\n        fleets = parse_fleets(raw_fleets)\n        my_planets = [p for p in planets if int(p.owner) == player]\n        targets = [p for p in planets if int(p.owner) != player]\n\n        moves = []\n        if not targets:\n            return moves\n\n        reserved_target_ids = set()\n        threatened_ids = threatened_planet_ids(player, planets, fleets)\n        production_owned = total_production(my_planets)\n        active_enemy_count = len(enemy_players(player, planets, fleets))\n        reinforcement_moves, reinforcement_sources = build_reinforcement_moves(\n            player,\n            my_planets,\n            planets,\n            fleets,\n            angular_velocity,\n        )\n        moves.extend(reinforcement_moves)\n        ordered_sources = sorted(my_planets, key=lambda p: (int(p.ships), int(p.production)), reverse=True)\n        for source in ordered_sources:\n            if int(source.id) in reinforcement_sources:\n                continue\n            if int(source.id) in threatened_ids:\n                continue\n            chosen = best_target(\n                source,\n                targets,\n                reserved_target_ids,\n                len(my_planets),\n                angular_velocity,\n                production_owned,\n                player,\n                planets,\n                fleets,\n                active_enemy_count,\n            )\n            if chosen is None:\n                continue\n            target, aim_target, ships = chosen\n            reserved_target_ids.add(int(target.id))\n            moves.append([int(source.id), launch_angle(source, aim_target), int(ships)])\n        return moves\n    except Exception:\n        return []\n'

CFG = {
    "agent_version": "roi_reserve_v5",
    "n_seeds": 30,
    "primary_opponent": "random",
    "fallback_opponent": "passive",
    "run_simulations": True,
    "write_submission": True,
}

IS_KAGGLE = Path("/kaggle").exists()
WORKING = Path("/kaggle/working") if IS_KAGGLE else Path("outputs/agent_submission")
WORKING.mkdir(parents=True, exist_ok=True)

AGENT_VERSION = str(CFG["agent_version"])
SOURCE_CANDIDATES = [
    Path("main.py"),
    Path("..") / "agents" / AGENT_VERSION / "main.py",
    Path("agents") / AGENT_VERSION / "main.py",
    Path("/kaggle/working/main.py"),
]


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    """Return the first existing path from ordered candidates."""
    for path in paths:
        if path.exists():
            return path
    return None


SOURCE_MAIN = first_existing(SOURCE_CANDIDATES)
SUBMISSION_MAIN = WORKING / "main.py"
if SOURCE_MAIN is None:
    if AGENT_VERSION != EMBEDDED_AGENT_VERSION:
        searched = "\n".join(str(path) for path in SOURCE_CANDIDATES)
        raise FileNotFoundError(f"Could not find agent source for {AGENT_VERSION}. Searched:\n{searched}")
    SUBMISSION_MAIN.write_text(EMBEDDED_AGENT_SOURCE)
elif SOURCE_MAIN.resolve() != SUBMISSION_MAIN.resolve():
    shutil.copy2(SOURCE_MAIN, SUBMISSION_MAIN)

print("is_kaggle:", IS_KAGGLE)
print("working:", WORKING)
print("agent_version:", AGENT_VERSION)
print("source_main:", SOURCE_MAIN)
print("submission_main:", SUBMISSION_MAIN)
print("cfg:", json.dumps(CFG, indent=2))


## 2. Static Verification

Compile the copied agent and run a deterministic action-format check before using the Kaggle environment. This catches the highest-risk submission failures early: missing `agent(obs)`, syntax errors, non-list actions, invalid move shape, non-integer ship counts, and impossible source ids.


In [ ]:
compile_result = None
try:
    compile(SUBMISSION_MAIN.read_text(), str(SUBMISSION_MAIN), "exec")
    compile_result = "ok"
except Exception:
    compile_result = traceback.format_exc()
    raise


def load_agent(path: Path) -> Any:
    """Load an agent module from a Python file."""
    spec = importlib.util.spec_from_file_location("submission_agent", path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Cannot load {path}")
    spec.loader.exec_module(module)
    return module


def run_synthetic_action_check(agent_module: Any) -> list:
    """Run a deterministic action-format smoke check."""
    obs = {
        "player": 0,
        "planets": [
            [0, 0, 10.0, 10.0, 1.0, 80, 5],
            [1, -1, 13.0, 10.0, 1.0, 12, 1],
            [2, -1, 24.0, 10.0, 2.0, 10, 5],
        ],
        "fleets": [],
        "angular_velocity": 0.03,
        "initial_planets": [],
        "comet_planet_ids": [],
    }
    moves = agent_module.agent(obs)
    assert isinstance(moves, list)
    for move in moves:
        assert isinstance(move, list)
        assert len(move) == 3
        assert isinstance(move[0], int)
        assert isinstance(move[1], float)
        assert isinstance(move[2], int)
        assert move[2] > 0
    return moves


agent_module = load_agent(SUBMISSION_MAIN)
synthetic_moves = run_synthetic_action_check(agent_module)
print("compile_result:", compile_result)
print("synthetic_moves:", synthetic_moves)


## 3. Kaggle Environment Check

Import Orbit Wars from `kaggle_environments` and record runtime metadata. Local runs may stop at static verification; Kaggle runs should confirm that the competition environment is available before simulations start.


In [ ]:
env_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "is_kaggle": IS_KAGGLE,
    "agent_version": AGENT_VERSION,
    "compile_result": compile_result,
    "synthetic_moves": synthetic_moves,
}
ENV_AVAILABLE = False
ENV_ERROR = None

try:
    import kaggle_environments
    from kaggle_environments import make

    ENV_AVAILABLE = True
    env_info["kaggle_environments_version"] = getattr(kaggle_environments, "__version__", "unknown")
except Exception as exc:
    ENV_ERROR = repr(exc)
    env_info["kaggle_environments_error"] = traceback.format_exc()

try:
    if ENV_AVAILABLE:
        make("orbit_wars", configuration={"seed": 0}, debug=True)
        env_info["orbit_wars_available"] = True
except Exception as exc:
    ENV_AVAILABLE = False
    ENV_ERROR = repr(exc)
    env_info["orbit_wars_available"] = False
    env_info["orbit_wars_error"] = traceback.format_exc()

print(json.dumps(env_info, indent=2, default=str))


## 4. Fixed-Seed Benchmark

Run the active agent against `random` over fixed seeds. The check focuses on **reproducibility**, **run errors**, and a simple win/loss signal; deeper strategic conclusions should come from replay review and the version log.


In [ ]:
PASSIVE_PATH = WORKING / "passive_agent.py"
PASSIVE_PATH.write_text(
    'def agent(obs):\n'
    '    """Return no actions for fallback benchmark simulations."""\n'
    '    return []\n'
)



def score_from_obs(obs: Any, player: int) -> float:
    """Calculate a final ship-count proxy for one player."""
    if isinstance(obs, dict):
        planets = obs.get("planets", []) or []
        fleets = obs.get("fleets", []) or []
    else:
        planets = getattr(obs, "planets", []) or []
        fleets = getattr(obs, "fleets", []) or []
    planet_score = sum(float(row[5]) for row in planets if int(row[1]) == player)
    fleet_score = sum(float(row[6]) for row in fleets if int(row[1]) == player)
    return planet_score + fleet_score


def summarize_seed(seed: int, env: Any, opponent_label: str) -> dict[str, Any]:
    """Summarize one completed Orbit Wars episode."""
    final_step = env.steps[-1]
    final_obs = final_step[0].observation
    return {
        "seed": seed,
        "opponent": opponent_label,
        "steps": len(env.steps),
        "player0_reward": final_step[0].reward,
        "player0_status": final_step[0].status,
        "player1_reward": final_step[1].reward if len(final_step) > 1 else None,
        "player1_status": final_step[1].status if len(final_step) > 1 else None,
        "player0_score_proxy": score_from_obs(final_obs, 0),
        "player1_score_proxy": score_from_obs(final_obs, 1),
    }


seed_rows = []
run_errors = []
N_SEEDS = int(CFG["n_seeds"])

if ENV_AVAILABLE and CFG["run_simulations"]:
    for seed in range(N_SEEDS):
        try:
            env = make("orbit_wars", configuration={"seed": seed}, debug=True)
            opponent_label = str(CFG["primary_opponent"])
            try:
                env.run([str(SUBMISSION_MAIN), opponent_label])
            except Exception:
                opponent_label = str(CFG["fallback_opponent"])
                env = make("orbit_wars", configuration={"seed": seed}, debug=True)
                env.run([str(SUBMISSION_MAIN), str(PASSIVE_PATH)])
            row = summarize_seed(seed, env, opponent_label)
            seed_rows.append(row)
            print("seed", seed, "reward", row["player0_reward"], "score_proxy", row["player0_score_proxy"])
        except Exception as exc:
            run_errors.append({"seed": seed, "error": repr(exc)})
            print("seed failed", seed, repr(exc))
else:
    print("Orbit Wars environment is unavailable in this runtime. Static verification only.")

seed_df = pd.DataFrame(seed_rows)
errors_df = pd.DataFrame(run_errors)
seed_df.to_csv(WORKING / "agent_submission_seed_summary.csv", index=False)
errors_df.to_csv(WORKING / "agent_submission_run_errors.csv", index=False)

env_info["seed_count_requested"] = N_SEEDS
env_info["seed_count_completed"] = len(seed_rows)
env_info["run_error_count"] = len(run_errors)
with open(WORKING / "agent_submission_environment_info.json", "w") as f:
    json.dump(env_info, f, indent=2, default=str)

wins = int((seed_df["player0_reward"] == 1).sum()) if not seed_df.empty else 0
losses = int((seed_df["player0_reward"] != 1).sum()) if not seed_df.empty else 0
win_rate = wins / len(seed_df) if len(seed_df) else 0.0
findings = f"""# Agent Submission Findings

- Agent version: `{AGENT_VERSION}`
- Orbit Wars environment available: `{ENV_AVAILABLE}`
- Seeds completed: `{len(seed_rows)}` of `{N_SEEDS}`
- Run errors: `{len(run_errors)}`
- Wins vs random: `{wins}`
- Losses vs random: `{losses}`
- Win rate vs random: `{win_rate:.1%}`
"""
(WORKING / "agent_submission_findings.md").write_text(findings)
print(findings)


## 5. Submission Package

Create a root-level `submission.tar.gz` containing `main.py`. Submit the archive from the notebook output only after the smoke benchmark is acceptable and `docs/04_agent_version_log.md` is ready to record the result.


In [ ]:
if CFG["write_submission"]:
    submission_tar = WORKING / "submission.tar.gz"
    with tarfile.open(submission_tar, "w:gz") as tar:
        tar.add(SUBMISSION_MAIN, arcname="main.py")
    print("wrote:", submission_tar)
else:
    print("write_submission is disabled")

print("wrote:", SUBMISSION_MAIN)
print("wrote:", WORKING / "agent_submission_environment_info.json")
print("wrote:", WORKING / "agent_submission_seed_summary.csv")
print("wrote:", WORKING / "agent_submission_run_errors.csv")
print("wrote:", WORKING / "agent_submission_findings.md")
